In [ ]:
from torchvision import transforms,datasets
from torch.utils.data import DataLoader
import torch.nn as nn
from pathlib import Path
import torch.optim as optim
import torchvision.models as models
import copy
import torch
import datetime
import sys
import os
from torch.utils.data import random_split
from collections import Counter
from sklearn.metrics import balanced_accuracy_score
sys.path.insert(0, os.path.abspath('..'))
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.insert(0, project_root)
from app.DATABASE.DB_FUNC import add_model


In [ ]:
# Для загрузки изображений
IMG_SIZE = 224
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
#формирование dataset'ов
path = Path('../data')

train_dataset  = datasets.ImageFolder(root=path / 'train', transform=train_transform)
val_dataset = datasets.ImageFolder(root=path / 'val', transform=test_transform)
test_dataset = datasets.ImageFolder(root=path / 'test', transform=test_transform)

#print(train_dataset.class_to_idx)

In [ ]:
#dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size = 32,
    shuffle=True,
    num_workers=8
)
val_loader = DataLoader(
    val_dataset,
    batch_size = 32,
    shuffle=False,
    num_workers=8
)
test_loader = DataLoader(
    test_dataset,
    batch_size = 32,
    shuffle=False,
    num_workers=8
)

In [ ]:
#модель
model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)

for param in model.parameters():
    param.requires_grad = False

num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 6)

for param in model.features[6].parameters():
    param.requires_grad = True
for param in model.features[7].parameters():
    param.requires_grad = True


In [ ]:

#ставим GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)


train_targets = [label for _, label in train_dataset]
class_counts = Counter(train_targets)
num_classes = len(train_dataset.classes)
total = sum(class_counts.values())
weights = [total / (num_classes * class_counts[i]) for i in range(num_classes)]
class_weights = torch.tensor(weights, dtype=torch.float).to(device)

loss_fn = nn.CrossEntropyLoss(weight=class_weights)


optimizer = torch.optim.Adam([
    {'params': model.classifier.parameters(), 'lr': 0.001},
    {'params': model.features[6].parameters(), 'lr': 0.0001},
    {'params': model.features[7].parameters(), 'lr': 0.0001}
],weight_decay=0.0001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='min',factor=0.5,
                                                       patience=4,threshold=0.0001,min_lr=0.000001)
#функция обучения
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    
    run_loss = 0.0
    corrects = 0
    total = 0
    
    for images, labels in loader:
        images,labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        run_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        corrects += (predicted == labels).sum().item()

    return run_loss / len(loader), 100 * corrects / total
#функция валидации
def validate(model, loader, loss_fn, device):
    model.eval()

    run_loss = 0.0
    corrects = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images,labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = loss_fn(outputs, labels)

            run_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            corrects += (predicted == labels).sum().item()

    return run_loss / len(loader), 100 * corrects / total

def compute_ba(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    ba = balanced_accuracy_score(all_labels, all_preds)
    return ba
    

In [ ]:
max_epochs = 50
best_acc = 0.0
best_loss = float('inf')
best_model_wts = copy.deepcopy(model.state_dict())
best_optimizer_state = copy.deepcopy(optimizer.state_dict())
best_epoch = 0
epochs_to_improve = 0
patience = 8
best_ba = 0
trained_model_path = f'C:/Users/pc/Desktop/project_cv/app/models/model_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}.pth'
for epoch in range(max_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, loss_fn, device)
    val_ba = compute_ba(model, val_loader, device)
    scheduler.step(val_loss)
    if val_ba > best_ba:
        best_ba = val_ba
        best_acc = val_acc
        best_loss = val_loss
        best_epoch = epoch + 1
        best_model_wts = copy.deepcopy(model.state_dict())
        best_optimizer_state = copy.deepcopy(optimizer.state_dict())
        epochs_to_improve = 0
    else:
        epochs_to_improve += 1
    if epochs_to_improve >= patience:
        print(f"Метрики не улучшались {patience} эпох. Прекращаем обучение")
        break
    print(f'Epoch {epoch + 1}/{max_epochs}, val_ba:{val_ba:.4f}, Train loss: {train_loss:.4f}, Train acc: {train_acc:.2f}%, Val loss: {val_loss:.4f}, Val acc: {val_acc:.2f}%')

model.load_state_dict(best_model_wts)
test_loss, test_acc = validate(model, test_loader, loss_fn, device)
print(f"\nточность на test выборке: {test_acc:.2f}%")
print(f"loss на test выборке: {test_loss:.4f}")
torch.save({
    'epoch': best_epoch,
    'model_state_dict': best_model_wts,
    'optimizer_state_dict': best_optimizer_state,
    'best_acc': best_acc,
    'val_loss': best_loss,
    'test_acc': test_acc,
    'test_loss': test_loss,
    'best_ba': best_ba,
    'class_to_idx': train_dataset.class_to_idx
}, trained_model_path)  
add_model(trained_model_path,best_acc, test_acc)
print(f"\nОбучение завершено. Лучшая точность на валидации: {best_acc:.2f}%")


In [ ]:

from torchinfo import summary

summary(model, input_size=(1, 3, 224, 224))  # batch_size=1

In [ ]:
from collections import Counter
train_labels = [label for _, label in train_dataset.samples]  # для ImageFolder
class_counts = Counter(train_labels)
print(class_counts)